<div class="markdown-google-sans">

<a name="machine-learning-examples"></a>

### Real-Time Zero-Day Attack Detection Using a Lightweight Cascade Architecture
Dataset: CSIC-2012

</div>


In [1]:
import pandas as pd
import numpy as np
import math
import time
import psutil
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

from scipy.sparse import hstack
from xgboost import XGBClassifier


# =========================================================
# Utility functions
# =========================================================

def entropy(s: str) -> float:
    if not s:
        return 0.0
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in probs)


def memory_mb() -> float:
    return psutil.Process().memory_info().rss / (1024 ** 2)


# =========================================================
# Stage 0 – Lightweight anomaly filter
# =========================================================

SUSPICIOUS_KEYWORDS = [
    "select", "union", "sleep(", "benchmark(",
    "<script", "../", "or 1=1", "--", "%27"
]

def stage0_filter(request: str,
                  entropy_thr=4.5,
                  length_thr=150,
                  keyword_hits=1) -> bool:
    """
    Returns True if request is suspicious
    """
    hits = sum(k in request.lower() for k in SUSPICIOUS_KEYWORDS)

    if entropy(request) > entropy_thr:
        return True
    if len(request) > length_thr:
        return True
    if hits >= keyword_hits:
        return True

    return False


# =========================================================
# Feature extraction
# =========================================================

def build_requests(df):
    headers = [
        "userAgent", "Pragma", "Cache-Control", "Accept",
        "Accept-encoding", "Accept-charset", "language",
        "host", "cookie", "content-type", "connection"
    ]

    reqs, payloads = [], []
    for _, r in df.iterrows():
        parts = [f"{r.get('Method','GET')} {r.get('URL','/')} HTTP/1.1"]
        for h in headers:
            v = r.get(h, "")
            if pd.notna(v) and v != "":
                parts.append(f"{h}: {v}")

        payload = r.get("payload", "") or r.get("content", "")
        payloads.append(str(payload))
        if payload:
            parts.append(f"Payload: {payload}")

        reqs.append("\n".join(parts))
    return reqs, payloads


def extract_features(reqs, payloads):
    feats = []
    for r, p in zip(reqs, payloads):
        L = len(r)
        feats.append([
            L,
            entropy(r),
            entropy(p),
            r.count("/"),
            r.count("&") + r.count("="),
            sum(c.isdigit() for c in r) / L if L else 0,
            sum(k in r.lower() for k in SUSPICIOUS_KEYWORDS)
        ])
    return np.array(feats)


# =========================================================
# Zero-day Detection System
# =========================================================

class ZeroDayWAF:
    def __init__(self):
        self.scaler = StandardScaler()
        self.vectorizer = TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            max_features=400,
            lowercase=True
        )

        # Stage-1: Recall-first
        self.stage1 = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=42
        )

        # Stage-2: Precision-first
        self.stage2 = XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            gamma=1.0,
            reg_alpha=0.5,
            reg_lambda=1.0,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=42
        )

        self.TH1 = 0.15  # recall-first
        self.TH2 = 0.65  # precision-first

    # -----------------------------------------------------

    def train(self, df, label_col="Label"):
        y = df[label_col].astype(int).values
        reqs, payloads = build_requests(df)

        stat = extract_features(reqs, payloads)
        stat_scaled = self.scaler.fit_transform(stat)
        tfidf = self.vectorizer.fit_transform(reqs)

        X = hstack([stat_scaled, tfidf]).tocsr()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, stratify=y, random_state=42
        )

        neg, pos = np.bincount(y_train)
        self.stage1.set_params(scale_pos_weight=neg / pos)

        print("🚀 Training Stage-1 (Recall-first)")
        self.stage1.fit(X_train, y_train)

        # Stage-2 training on suspicious samples
        scores = self.stage1.predict_proba(X_train)[:, 1]
        mask = scores > self.TH1

        print("🚀 Training Stage-2 (Precision-first)")
        self.stage2.fit(X_train[mask], y_train[mask])

        self.X_test = X_test
        self.y_test = y_test
        self.reqs_test = reqs

    # -----------------------------------------------------

    def predict(self):
        scores1 = self.stage1.predict_proba(self.X_test)[:, 1]
        suspicious = scores1 > self.TH1

        y_pred = np.zeros_like(self.y_test)
        scores2 = self.stage2.predict_proba(self.X_test[suspicious])[:, 1]
        y_pred[suspicious] = (scores2 > self.TH2).astype(int)

        return y_pred

    # -----------------------------------------------------

    def evaluate(self):
        mem_before = memory_mb()
        t0 = time.time()
        y_pred = self.predict()
        infer_time = (time.time() - t0) * 1000
        mem_after = memory_mb()

        cm = confusion_matrix(self.y_test, y_pred)
        acc = np.mean(self.y_test == y_pred)
        fpr = cm[0, 1] / (cm[0, 0] + cm[0, 1])

        latency = infer_time / self.X_test.shape[0]
        throughput = 1000 / latency

        print("\n📊 ZERO-DAY DETECTION RESULTS")
        print(f"Accuracy     : {acc:.4f}")
        print(f"FPR          : {fpr:.4f}")
        print(f"Latency      : {latency:.4f} ms")
        print(f"Throughput   : {throughput:.0f} req/s")
        print(f"Memory (MB)  : {mem_after - mem_before:.2f}")
        print("Confusion Matrix:")
        print(cm)


# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":
    DATA_PATH = "/content/sample_data/full_csic_test.csv"

    print("🔄 Loading dataset...")
    df = pd.read_csv(DATA_PATH)

    waf = ZeroDayWAF()
    waf.train(df, label_col="Label")
    waf.evaluate()


🔄 Loading dataset...
🚀 Training Stage-1 (Recall-first)
🚀 Training Stage-2 (Precision-first)

📊 ZERO-DAY DETECTION RESULTS
Accuracy     : 0.8922
FPR          : 0.0101
Latency      : 0.0177 ms
Throughput   : 56490 req/s
Memory (MB)  : 0.00
Confusion Matrix:
[[10691   109]
 [ 1865  5655]]
